# 02. LLM 파인튜닝 템플릿 — QLoRA / DoRA (12GB)
4비트 NF4 양자화 + LoRA(또는 DoRA) + 8비트 페이지드 Adam + Gradient checkpointing.
- 기본 모델은 빠른 실습용 0.5B. `MODEL_ID` 를 7B~13B 로 바꿔도 12GB 에서 돌아가도록 구성했습니다 (배치·시퀀스 길이는 줄이세요).
- 모델·데이터는 `HF_HOME`(영구 저장 위치)에 캐시되어 다시 받지 않습니다.

In [ ]:
import os, torch, transformers, peft, bitsandbytes as bnb
print("transformers", transformers.__version__, "| peft", peft.__version__, "| bitsandbytes", bnb.__version__)
print("HF_HOME =", os.environ.get("HF_HOME"))
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"   # 예: "Qwen/Qwen2.5-7B-Instruct"
USE_DORA = False                           # True 로 바꾸면 DoRA (조금 느리지만 품질↑)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
bnb_cfg = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                             bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.bfloat16)
tok = AutoTokenizer.from_pretrained(MODEL_ID)
tok.pad_token = tok.pad_token or tok.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb_cfg,
                                             torch_dtype=torch.bfloat16, device_map={"": 0},
                                             attn_implementation="sdpa")
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, task_type="CAUSAL_LM", use_dora=USE_DORA,
                  target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"])
model = get_peft_model(model, lora)
model.print_trainable_parameters()
print(f"VRAM 사용: {torch.cuda.memory_allocated()/2**30:.2f} GB")

In [ ]:
# 실습용 소형 데이터 (실제로는 datasets.load_dataset 으로 교체)
from datasets import Dataset
pairs = [("인테리어 견적서에 꼭 들어가야 할 항목은?", "공종, 자재 규격, 수량, 단가, 인건비, 부가세, 공사 기간입니다."),
         ("분전반 과열의 흔한 원인은?", "단자 풀림, 과부하, 접촉 불량, 노후 차단기입니다."),
         ("UTP 케이블 T568B 순서는?", "주황백, 주황, 녹백, 파랑, 파백, 녹, 갈백, 갈입니다.")] * 40
def fmt(q, a):
    msgs = [{"role": "user", "content": q}, {"role": "assistant", "content": a}]
    return tok.apply_chat_template(msgs, tokenize=False)
ds = Dataset.from_list([{"text": fmt(q, a)} for q, a in pairs])
ds = ds.map(lambda b: tok(b["text"], truncation=True, max_length=256), batched=True, remove_columns=["text"])
print(ds)

In [ ]:
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling
args = TrainingArguments(
    output_dir="/tmp/qlora-out", per_device_train_batch_size=4, gradient_accumulation_steps=4,
    max_steps=30, learning_rate=2e-4, lr_scheduler_type="cosine", warmup_steps=3,
    bf16=True, optim="paged_adamw_8bit",              # 옵티마이저 상태 VRAM 절반 이하
    gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False},
    logging_steps=5, save_strategy="no", report_to=[], dataloader_pin_memory=True)
trainer = Trainer(model=model, args=args, train_dataset=ds,
                  data_collator=DataCollatorForLanguageModeling(tok, mlm=False))
trainer.train()
print(f"최대 VRAM: {torch.cuda.max_memory_allocated()/2**30:.2f} GB")

In [ ]:
model.eval()
msgs = [{"role": "user", "content": "분전반 과열의 흔한 원인은?"}]
ids = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt").to(0)
with torch.no_grad():
    out = model.generate(ids, max_new_tokens=64, do_sample=False)
print(tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True))
model.save_pretrained(os.path.join(os.environ.get("HF_HOME", "/tmp"), "adapters", "demo-lora"))